# Bizarro World — Gemma 2B behavioral friction (Colab)

**Runtime:** `Runtime` → `Change runtime type` → **GPU** (T4 or better).

**Hugging Face:** Gemma is gated. In Colab: `Secrets` (key icon) → add `HF_TOKEN` with a [read token](https://huggingface.co/settings/tokens). Accept the Gemma license on the model card first.

First run downloads weights (~few GB) then runs two short forward passes; GPU uses float16.

In [ ]:
# Installs (Colab often has torch+CUDA already; TransformerLens tracks recent torch)
%pip install -q transformer-lens huggingface_hub

In [ ]:
import os
from typing import Any, Dict, List

import torch
from huggingface_hub import login
from transformer_lens import HookedTransformer

# Colab: load HF token from Secrets so gated Gemma can download
try:
    from google.colab import userdata

    try:
        _tok = userdata.get("HF_TOKEN")
    except Exception:
        _tok = None
        print("Colab: add Secret named HF_TOKEN (or run huggingface-cli login locally).")
    if _tok:
        login(token=_tok, add_to_git_credential=False)
except ImportError:
    pass  # local Jupyter: run `huggingface-cli login` or set HF_TOKEN in env

if not os.environ.get("HF_TOKEN") and os.environ.get("HUGGING_FACE_HUB_TOKEN"):
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"]

MODEL_NAME = "google/gemma-2b"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("device:", device, "dtype:", dtype)
if device == "cpu":
    print("Warning: no GPU; enable GPU in Runtime settings for faster runs.")

In [ ]:
def load_model() -> HookedTransformer:
    model = HookedTransformer.from_pretrained(
        MODEL_NAME,
        device=device,
        dtype=dtype,
    )
    model.eval()
    return model


model = load_model()
print("Loaded", MODEL_NAME)

In [ ]:
def _token_ids_for_target(target_token: str) -> List[int]:
    """Gemma uses SentencePiece; verify single-token targets."""
    ids = model.tokenizer.encode(target_token, add_special_tokens=False)
    if isinstance(ids, int):
        ids = [ids]
    return list(ids)


def _softmax_entropy_from_logits(final_logits: torch.Tensor) -> torch.Tensor:
    probs = torch.softmax(final_logits, dim=-1)
    log_probs = torch.log(probs + 1e-20)
    return -(probs * log_probs).sum()


def analyze_behavioral_friction(
    clean_prompt: str,
    bizarro_prompt: str,
    target_token: str,
) -> Dict[str, Any]:
    target_ids = _token_ids_for_target(target_token)
    if len(target_ids) != 1:
        raise ValueError(
            f"`target_token={target_token!r}` encodes to {len(target_ids)} ids {target_ids}. "
            "Expected a single token id."
        )
    target_id = target_ids[0]

    def run_one(prompt: str) -> Dict[str, float]:
        tokens = model.to_tokens(prompt, prepend_bos=True).to(model.cfg.device)
        with torch.no_grad():
            logits = model(tokens)
        final_logits = logits[0, -1, :]
        probs = torch.softmax(final_logits, dim=-1)
        prob_target = probs[target_id].item()
        entropy = _softmax_entropy_from_logits(final_logits).item()
        return {"prob": prob_target, "entropy": entropy}

    clean_stats = run_one(clean_prompt)
    bizarro_stats = run_one(bizarro_prompt)
    print(f"Target token {target_token!r} (id={target_id}):")
    print(f"  Clean   prob={clean_stats['prob']:.6g} entropy={clean_stats['entropy']:.6g}")
    print(f"  Bizarro prob={bizarro_stats['prob']:.6g} entropy={bizarro_stats['entropy']:.6g}")
    return {
        "target_token": target_token,
        "target_token_id": target_id,
        "clean": clean_stats,
        "bizarro": bizarro_stats,
    }

In [ ]:
clean_prompt = (
    "The capital of France is Paris. The shape of a standard basketball is a"
)
bizarro_prompt = (
    "In Bizarro World, everything round becomes a cube. The shape of a standard basketball is a"
)

target_tokens = [" sphere", " cube"]

for t in target_tokens:
    print(f"{t!r} -> ids {_token_ids_for_target(t)}")

results = []
for t in target_tokens:
    results.append(analyze_behavioral_friction(clean_prompt, bizarro_prompt, t))

results

## Next steps (README roadmap)

1. **Activation patching / causal tracing** — Find components that push "sphere" on the factual prompt, then patch those activations into the Bizarro forward pass and measure whether logits flip back (epistemic override vs context-only suppression).
2. **Narrow the search** — Start with late-layer MLPs and induction-adjacent heads; sweep a small grid before full residual stream patches.
3. **Metrics** — Logit difference on ` sphere` vs ` cube`, entropy at the decisive position, and optionally KL between clean vs patched runs.
4. **Persistence** — Optional: set `HF_HOME` to Google Drive so the model cache survives disconnects; keep large activation dumps off `/content` or delete between sweeps to stay under disk limits.

When you are ready to implement patching in-repo, we can add a second notebook or Python module that reuses `model` hooks from TransformerLens.